In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os
os.chdir('/content/drive/MyDrive/RF_2025')
os.getcwdb()

b'/content/drive/MyDrive/RF_2025'

In [1]:
!pip install facenet_pytorch

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.7 kB)
  Using cached triton-2.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.

In [2]:
pip install --upgrade torch torchvision

  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_cu12-11.7.3.90-py3-none-manyl

In [3]:
 # Chargement des images
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder('/content/drive/MyDrive/RF_2025/Data/A uploader/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = datasets.ImageFolder('/content/drive/MyDrive/RF_2025/Data/A uploader/val', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [4]:
#Charger le modèle FaceNet
from facenet_pytorch import InceptionResnetV1

model = InceptionResnetV1(pretrained='vggface2', classify=False, device='cuda').to('cuda')
model.train()

# Option : Geler certaines couches
for param in list(model.parameters())[:-10]:
    param.requires_grad = False

  0%|          | 0.00/107M [00:00<?, ?B/s]

In [5]:
import random

def create_triplets(embeddings, labels):
    anchors, positives, negatives = [], [], []
    labels = labels.cpu().numpy()

    label_to_indices = {}
    for idx, label in enumerate(labels):
        if label not in label_to_indices:
            label_to_indices[label] = []
        label_to_indices[label].append(idx)

    all_labels = list(label_to_indices.keys())

    for label in all_labels:
        pos_indices = label_to_indices[label]
        if len(pos_indices) < 2:
            continue  # il faut au moins deux images pour faire (anchor, positive)

        neg_labels = [l for l in all_labels if l != label]
        if not neg_labels:
            continue  # il faut au moins une classe différente

        for _ in range(min(2, len(pos_indices) - 1)):  # max 2 triplets par classe
            a, p = random.sample(pos_indices, 2)
            n_label = random.choice(neg_labels)
            n = random.choice(label_to_indices[n_label])

            anchors.append(embeddings[a])
            positives.append(embeddings[p])
            negatives.append(embeddings[n])

    if len(anchors) == 0:
        return None, None, None  # rien à entraîner sur ce batch
    return torch.stack(anchors), torch.stack(positives), torch.stack(negatives)

In [6]:
# Définir la triplet loss
import torch.nn.functional as F
import torch.nn as nn
import torch.nn.functional as F
import torchvision

def triplet_loss(anchor, positive, negative, alpha=0.2):
    pos_dist = F.pairwise_distance(anchor, positive)
    neg_dist = F.pairwise_distance(anchor, negative)
    loss = torch.mean(torch.clamp(pos_dist - neg_dist + alpha, min=0.0))
    return loss

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

# Boucle
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    skipped = 0

    for imgs, labels in train_loader:
        imgs = imgs.to('cuda')
        labels = labels.to('cuda')

        embeddings = model(imgs)
        anchor, positive, negative = create_triplets(embeddings, labels)

        if anchor is None:
            skipped += 1
            continue

        loss = triplet_loss(anchor, positive, negative)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f} - Batches Skipped: {skipped}")


Epoch 1/50 - Loss: 2.8140 - Batches Skipped: 0
Epoch 2/50 - Loss: 2.4256 - Batches Skipped: 0
Epoch 3/50 - Loss: 2.0389 - Batches Skipped: 0
Epoch 4/50 - Loss: 1.7025 - Batches Skipped: 0
Epoch 5/50 - Loss: 1.9816 - Batches Skipped: 0
Epoch 6/50 - Loss: 1.5541 - Batches Skipped: 0
Epoch 7/50 - Loss: 1.4282 - Batches Skipped: 0
Epoch 8/50 - Loss: 1.7458 - Batches Skipped: 0
Epoch 9/50 - Loss: 1.2067 - Batches Skipped: 0
Epoch 10/50 - Loss: 1.1791 - Batches Skipped: 0
Epoch 11/50 - Loss: 1.4534 - Batches Skipped: 0
Epoch 12/50 - Loss: 1.0866 - Batches Skipped: 1
Epoch 13/50 - Loss: 0.6983 - Batches Skipped: 0
Epoch 14/50 - Loss: 1.0459 - Batches Skipped: 0
Epoch 15/50 - Loss: 1.1837 - Batches Skipped: 0
Epoch 16/50 - Loss: 1.0367 - Batches Skipped: 0
Epoch 17/50 - Loss: 1.0286 - Batches Skipped: 1
Epoch 18/50 - Loss: 0.7858 - Batches Skipped: 0
Epoch 19/50 - Loss: 1.2593 - Batches Skipped: 0
Epoch 20/50 - Loss: 0.6538 - Batches Skipped: 0
Epoch 21/50 - Loss: 0.8253 - Batches Skipped: 1
E

In [9]:
#Évaluation simple sur les visages
from torch.nn.functional import cosine_similarity

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to('cuda')
        embeddings = model(imgs)

        # Compare les 2 premiers visages du batch par exemple :
        sim = cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0))
        print(f"Cosine similarity: {sim.item():.4f}")

Cosine similarity: 0.9110
Cosine similarity: 0.9414
Cosine similarity: 0.7618
Cosine similarity: 0.9574
Cosine similarity: 0.8763
Cosine similarity: 0.9189
Cosine similarity: 0.5033
Cosine similarity: 0.6590
Cosine similarity: 0.7091
Cosine similarity: 0.9751
Cosine similarity: 0.4137
Cosine similarity: 0.8613
Cosine similarity: 0.9492
Cosine similarity: 0.7702
Cosine similarity: 0.9332
Cosine similarity: 0.2440
Cosine similarity: 0.9647
Cosine similarity: 0.9587
Cosine similarity: 0.7205
Cosine similarity: 0.9793
Cosine similarity: 0.2136
Cosine similarity: -0.4483
Cosine similarity: 0.8826
Cosine similarity: 0.8190
Cosine similarity: 0.8131
Cosine similarity: 0.6847


In [14]:
# Sauvegarde du modèle fine-tuné
torch.save(model.state_dict(), "facenet_africain_djorod.pth")

In [15]:
from google.colab import files
files.download("/content/drive/MyDrive/RF_2025/facenet_africain_djorod.pth")

FileNotFoundError: Cannot find file: /content/drive/MyDrive/RF_2025/facenet_africain_djorod.pth

In [ ]:
!pip install gdown

To use `gdown`, you first need to get the file ID of `facenet_africaned_finetunedepoch50.pth` from your Google Drive. You can find this in the file's shareable link. The ID is the string of characters after `/d/` and before `/edit` or `/view`.

Replace `YOUR_FILE_ID_HERE` in the code below with the actual file ID.

In [ ]:
!gdown --id YOUR_FILE_ID_HERE -O facenet_africain_finetunedepoch50.pth

In [16]:
from google.colab import files
files.download("facenet_africain_djorod.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>